# Inspect a small CPU training run

Train a bounded synthetic regression example, inspect its recorded diagnostics, and reopen the retained run.
This demonstrates artifact and widget mechanics, not predictive quality or improvement advice.
Use an environment with this candidate Kumo Engine build and CPU runtime dependencies installed.
The [Engine setup guide](../docs/development/setup.md) describes source environments.
No cloud credentials are required.


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from kumoapi.model_plan import ModelPlan

import kumo.engine as kumo

# Keep this directory to reopen the diagnostics after the kernel exits.
demo_dir = Path(os.environ.get("KUMO_DEMO_DIR", "kumo-training-demo")).resolve()
data_dir = demo_dir / "data"
data_dir.mkdir(parents=True, exist_ok=True)
kumo.configure(scratch_dir=demo_dir / "scratch")
torch.set_num_threads(2)


In [ ]:
seed = 20261103
rng = np.random.default_rng(seed)
x = rng.normal(size=400)
pd.DataFrame({
    "ID": np.arange(400),
    "Y": 0.5 * x + rng.normal(scale=0.1, size=400),
    "F1": x,
}).to_parquet(data_dir / "entities.parquet", index=False)

connector = kumo.FileConnector(root_dir=str(data_dir))
entities = kumo.Table.from_source_table(
    connector["entities.parquet"], primary_key="ID"
).infer_metadata()
graph = kumo.Graph(tables={"entities": entities}, edges=[])
snapshot = graph.snapshot()
query = kumo.PredictiveQuery(
    graph=graph, query="PREDICT entities.Y FOR EACH entities.ID"
)


In [ ]:
plan = ModelPlan()
plan.training_job.num_experiments = 1
plan.training_job.manual_seed = seed
plan.training_job.refit_trainval = False
plan.training_job.refit_full = False
plan.training_job.disable_explain = False
plan.optimization.max_epochs = 3
plan.optimization.min_steps_per_epoch = 1
plan.optimization.max_steps_per_epoch = 2
plan.optimization.max_val_steps = 2
plan.optimization.max_test_steps = 2
plan.optimization.batch_size = [64]

trainer = kumo.Trainer(model_plan=plan, accelerator="cpu")
result = trainer.fit(query.generate_training_table(snapshot), snapshot=snapshot)
run_root = result.explanations.run_root
(demo_dir / "retained-run.txt").write_text(run_root, encoding="utf-8")
print("Retain this run directory:", run_root)


## Read the recorded results

Find the TEST metric and its fit, inspect the strongest recorded input sensitivity, then open its cohorts and a selected example.
The widget preloads at most 12 examples within byte limits for both notebook transports.
Selection strategies may overlap; they are not severity labels.
Example values and identities enter saved notebook output.
Use `max_samples=0` to omit example bodies.


In [ ]:
display(result.explanations.visualize(max_samples=12))


## Reopen without training

Run these cells in a later session after setting `demo_dir` to the same directory.
Keep the entire training run, not only the published model bundle.
The widget reads local retained files and does not run prediction, retraining, or explanations for new entities.


In [ ]:
from kumo.engine import TrainingExplanations

retained = (demo_dir / "retained-run.txt").read_text(encoding="utf-8")
reopened = TrainingExplanations.load(retained)
display(reopened.visualize(max_samples=12))


This example has one task, one trial, no refit, and no relationships.
It does not exercise classification, link prediction, architecture diversity, or native cloud notebook behavior.
A missing artifact remains unavailable; it is never shown as zero.
